In [1]:
import pandas as pd
import numpy as np
import faiss
import requests

In [2]:
index = faiss.read_index('index')

In [3]:
df = pd.read_csv('cz_real_estate.csv')

In [4]:
property_of_interest = df[df.location.str.contains('Roztoky u Prahy') & 
   df.title.str.contains('pronájem|pronajem', case=False)]

In [5]:
property_of_interest

,title,price,currency,location,description_en,description_native,construction_of_building,condition,equipped,area_of_property,...,age,garage,elevator,balcony,parking,barrier_free_access,cellar,near_public_transport,terrace,transaction_type
8057,Pronájem domu 3+1 • 70 m² bez realitky Za poto...,30000.0,CZK,"Za potokem, Roztoky - Roztoky u Prahy, Středoč...",I am offering a family house with a layout of ...,Nabízím k pronájmu rodinný dům o dispozici 3+1...,Smíšená,Dobrý,Částečně,600.0,...,NaN,0,0,0,1,0,1,0,0,rent


In [6]:
def create_textual_representation(row):
    textual_representation = f""" Title: {row["title"]},
Price: {row["price"]} {row["currency"]},

Description_in_native: {row["description_native"]},

Description_in_english: {row["description_en"]}""" # Let's test these features for now
    return textual_representation

In [7]:
df['textual_representation'] = df.apply(create_textual_representation, axis=1)
representation = df['textual_representation'].iloc[8057]

In [8]:
type(representation)

str

In [9]:
res = requests.post('http://localhost:11434/api/embeddings',
                    json={
                        'model': 'nomic-embed-text',
                        'prompt': representation
                    }
                   )

embedding = np.array([res.json()['embedding']], dtype = 'float32')

D, I = index.search(embedding, 5)

In [10]:
D

array([[ 0.      , 60.108437, 62.599186, 64.89725 , 65.82964 ]],
      dtype=float32)

In [11]:
I

array([[8057, 2434, 3738, 8320, 1771]])

In [12]:
best_matches = np.array(df['textual_representation'])[I.flatten()]

for match in best_matches[1:]: # skip the first (it's identical to the query)
    print('NEXT PROPERTY')
    print(match)
    print()


NEXT PROPERTY
 Title: Pronájem bytu 3+kk • 65 m² bez realitky Palackého třída, Brno - Královo Pole, Jihomoravský kraj,
Price: 10000.0 CZK,

Description_in_native: Nabízíme pokoj v RD v Králově poli.
Jedná se o samostatný pokoj v 1. patře rodinného domu. Na patře se dále nachází další dva pokoje a dva spolubydlící se kterými je společná kuchyň a koupelna. Na bytě se nachází pračka, ale pokoj je nevybavený. Celý dům nedávno prošel rekonstrukcí, takže vše to tam ještě voní novotou.
K dispozici je také prostorná nádherná terasa, která přímo vybízí k odpočinku.
Dům se nachází v klidné lokalitě ale zároveň 2 min chůze od zastávky MHD a 10 min do centra.
V případě zájmu mě neváhejte kontaktovat.,

Description_in_english: We offer a room in a family house in Královo Pole.
It is an independent room on the first floor of the house. On this floor, there are two more rooms and two housemates with whom the kitchen and bathroom are shared. Although there is a washing machine in the apartment, the ro

# Discussion
It seems that we are getting random recommendations.
Let's try to change query and see if the matches will be close.

## Different country

In [13]:
property_of_interest = df[df.location.str.contains('Berlin') &
  df.title.str.contains('pronájem|pronajem', case=False)]
property_of_interest.iloc[:5]

,title,price,currency,location,description_en,description_native,construction_of_building,condition,equipped,area_of_property,...,garage,elevator,balcony,parking,barrier_free_access,cellar,near_public_transport,terrace,transaction_type,textual_representation
36,Inzerát již není v nabídce Pronájem bytu 2+1 •...,1813.0,EUR,Berlin Mitte Berlin 10119,NaN,Tento atraktivní 2-pokojový byt s přibližně 70...,NaN,Po rekonstrukci,NaN,NaN,...,0,1,0,0,0,0,1,0,rent,Title: Inzerát již není v nabídce Pronájem by...
175,Pronájem bytu 2+1 • 55 m² bez realitky Einödsh...,1600.0,EUR,Einödshofer Weg Berlin Mariendorf Berlin 12109,NaN,Tento vysoce kvalitní podkrovní byt v 5. patře...,NaN,Po rekonstrukci,Vybaveno,NaN,...,0,0,1,0,0,1,1,0,rent,Title: Pronájem bytu 2+1 • 55 m² bez realitky...
228,Pronájem bytu 3+1 • 76 m² bez realitky Berlin ...,1500.0,EUR,Berlin Frohnau Berlin 13465,NaN,Tento 3-pokojový byt o přibližně 75 m² obytné ...,NaN,NaN,NaN,NaN,...,1,0,1,0,0,1,1,0,rent,Title: Pronájem bytu 3+1 • 76 m² bez realitky...
255,Pronájem bytu • 74 m² bez realitky Grünauer St...,550.0,EUR,Grünauer Straße 54 Berlin Köpenick Berlin 12557,NaN,Světlý a moderně vybavený byt v prvním patře b...,NaN,NaN,Vybaveno,NaN,...,0,0,1,0,0,1,1,0,rent,Title: Pronájem bytu • 74 m² bez realitky Grü...
358,Pronájem bytu 2+1 • 48 m² bez realitky Gélieus...,1200.0,EUR,Gélieustraße 7a Berlin Lichterfelde Berlin 12203,NaN,ZÁKLADNÍ NÁKRES NENÍ VHODNÝ PRO SDÍLENÉ BYDLET...,NaN,Po rekonstrukci,NaN,NaN,...,0,1,1,0,0,1,1,1,rent,Title: Pronájem bytu 2+1 • 48 m² bez realitky...


In [14]:
df['textual_representation'] = df.apply(create_textual_representation, axis=1)
representation = df['textual_representation'].iloc[36]

res = requests.post('http://localhost:11434/api/embeddings',
                    json={
                        'model': 'nomic-embed-text',
                        'prompt': representation
                    }
                   )

embedding = np.array([res.json()['embedding']], dtype = 'float32')

D, I = index.search(embedding, 5)

best_matches = np.array(df['textual_representation'])[I.flatten()]

for match in best_matches[1:]: # skip the first (it's identical to the query)
    print('NEXT PROPERTY')
    print(match)
    print()


NEXT PROPERTY
 Title: Pronájem bytu 3+1 • 68 m² bez realitky Parkstrasse 3, , Hessen,
Price: 1290.0 EUR,

Description_in_native: Tato exkluzivní 3-pokojová bytová jednotka s přibližně 68 m² obytné plochy se nachází ve 2. patře udržovaného činžovního domu ve velmi centrální lokalitě v Bad Vilbel. Byt je k dispozici od 1. března 2026. Bydlení projde kompletní rekonstrukcí a díky svému prvotřídnímu interiérovému řešení nabízí příjemné bydlení. Byt je pronajímán nezařízený.
Bad Vilbel, známý jako lázeňské město, nabízí atraktivní lokalitu k bydlení s dobrou infrastrukturou. Město disponuje řadou nákupních možností, lékařskou péčí a restauracemi. Veřejná doprava je dobře zajištěna, s pravidelnými spoji do okolních měst i do Frankfurtu. Parky a zelené plochy lákají k relaxaci a poskytují řadu volnočasových aktivit.
Infrastruktura (v okruhu 5 km):
Lékárna, diskont potravin, praktický lékař, mateřská škola, základní škola, Hauptschule, Realschule, gymnázium, Gesamtschule, veřejná doprava
Byt b

In [15]:
df['textual_representation'] = df.apply(create_textual_representation, axis=1)
representation = df['textual_representation'].iloc[358]

res = requests.post('http://localhost:11434/api/embeddings',
                    json={
                        'model': 'nomic-embed-text',
                        'prompt': representation
                    }
                   )

embedding = np.array([res.json()['embedding']], dtype = 'float32')

D, I = index.search(embedding, 5)

best_matches = np.array(df['textual_representation'])[I.flatten()]

for match in best_matches[1:]: # skip the first (it's identical to the query)
    print('NEXT PROPERTY')
    print(match)
    print()


NEXT PROPERTY
 Title: Pronájem bytu 2+1 • 55 m² bez realitky Čapkova, Karviná - Karviná-město, Moravskoslezský kraj,
Price: 7450.0 CZK,

Description_in_native: Nabízíme k pronájmu exkluzivní byt po kompletní rekonstrukci, který se nachází ve čtvrtém patře cihlového domu bez výtahu, přímo naproti parku. Tento světlý a stylový byt je jako stvořený pro mladý pár, který hledá moderní bydlení s výbornou dostupností i atmosférou. Dominantou bytu je hlavní pokoj s francouzským oknem a přímým vstupem na balkon, odkud si můžete vychutnávat klidný výhled do zeleně. V celém bytě jsou položeny kvalitní plovoucí podlahy, které dodávají prostoru elegantní a čistý vzhled. Kuchyně je vybavena moderní kuchyňskou linkou a nabízí dostatek místa na další vybavení podle vašich potřeb. Praktickým bonusem je komora s přípojkou na pračku, ideální jako prádelna nebo úložný prostor. Koupelna je vybavena vanou, toaleta je oddělená – což přináší větší komfort každodenního bydlení. V chodbě je prostor pro šatní sk

In [16]:
df['textual_representation'] = df.apply(create_textual_representation, axis=1)
representation = df['textual_representation'].iloc[175]

res = requests.post('http://localhost:11434/api/embeddings',
                    json={
                        'model': 'nomic-embed-text',
                        'prompt': representation
                    }
                   )

embedding = np.array([res.json()['embedding']], dtype = 'float32')

D, I = index.search(embedding, 5)

best_matches = np.array(df['textual_representation'])[I.flatten()]

for match in best_matches[1:]: # skip the first (it's identical to the query)
    print('NEXT PROPERTY')
    print(match)
    print()


NEXT PROPERTY
 Title: Pronájem bytu 5+1 • 155 m² bez realitky Merseburger Str. 406 Halle/Saale Ammendorf Sachsen-Anhalt 06132,
Price: 1850.0 EUR,

Description_in_native: Prostorný a plně zařízený 5-pokojový byt s obytnou plochou 155 m² nabízí ideální podmínky pro montéry, pracovníky na dohodu, expaty nebo pro společné bydlení.
Byt se nachází v Halle-Ammendorfu a vyniká skvělou dopravní dostupností do Halle i do Schkeuditzu směrem k DHL Hub.
5-pokojový byt se nachází v Halle-Ammendorfu, v klidné rezidenční oblasti s rychlým spojením. Možnost nakupování je v bezprostřední blízkosti.
Lokalita nabízí velmi dobrou dostupnost do Halle, do Schkeuditzu (například ke DHL Hub) a do Merseburgu.
Jsou k dispozici další parkovací místa pro automobily.
5 prostorných místností – flexibilně využitelných jako ložnice, pracovny či společné prostory, až s možností ubytovat 8 postelí.
2 moderní koupelny – jedna s vanou, druhá se sprchou, obě s vlastní pračkou.
Vestavěná kuchyň – plně vybavená pro okamžité 

# Discussion
It seems that there is some intelligence in the match - only one match is from CZ, the rest are from Germany.

Let's try to make a clear query

### Clear query

In [20]:
representation = "3+1 byt k pronájmu Praha, 15000 CZK, rekonstruovaný, rent, Czech Republic, Prague"

res = requests.post('http://localhost:11434/api/embeddings',
                    json={
                        'model': 'nomic-embed-text',
                        'prompt': representation
                    }
                   )

embedding = np.array([res.json()['embedding']], dtype = 'float32')

D, I = index.search(embedding, 10)

best_matches = np.array(df['textual_representation'])[I.flatten()]

for match in best_matches[1:]: # skip the first (it's identical to the query)
    print('NEXT PROPERTY')
    print(match)
    print()

NEXT PROPERTY
 Title: Pronájem bytu • 80 m² bez realitky Pštrossova, Praha - Nové Město,
Price: 112041.0 CZK,

Description_in_native: nan,

Description_in_english: Unique 3-bedroom apartment by the National theater. Luxury Pštrossova apartment for rent just a few steps from Wenceslas Square. Renovated apartment in the very center of Prague.
IMPORTANT INFORMATION!
Please be advised that due to legal regulations we are required to register you with the foreign police and collect city tax upon your arrival.
The city tax is 2,10 EUR per person per day. Children under 18 and people from Prague do not pay this fee.
Living in this area offers many cultural activities, as well as easy access to public transport and walking distance to many shops (Quadrio), theaters and cinemas. A few meters from the apartment there is the Můstek and Národní třída metro station.
The apartment consists of:
-Fast Wi-Fi
-Kitchen with full equipment
-3 Bathrooms
-3 Bedrooms

NEXT PROPERTY
 Title: Pronájem bytu 3+1 

# Discussion
The third match looks completely random.

Let's see if query format affects the search.

## Query format

In [33]:
representation = "3+1 byt k pronájmu Praha, 15000 CZK, rekonstruovaný, rent, Czech Republic, Prague"

res = requests.post('http://localhost:11434/api/embeddings',
                    json={'model': 'nomic-embed-text', 'prompt': representation})
query_embedding = np.array(res.json()['embedding'], dtype='float32').reshape(1, -1)

D, I = index.search(query_embedding, k=5)
print(D)

[[185.6018  215.19856 218.41849 221.16122 224.54181]]


In [34]:
representation = """Title: Pronájem bytu 3+1, Praha,
Price: 15000 CZK,
Description_in_english: 2 bedroom apartment for rent in Prague, renovated, close to metro, parking available"""

res = requests.post('http://localhost:11434/api/embeddings',
                    json={'model': 'nomic-embed-text', 'prompt': representation})
query_embedding = np.array(res.json()['embedding'], dtype='float32').reshape(1, -1)

D, I = index.search(query_embedding, k=5)
print(D)

[[137.78725 138.3623  140.6383  141.6314  142.3622 ]]


# Discussion
It seems that proper query format improves the search, but not sure if this is a valid way to interpret it.

Maybe descriptions confuse the embedding model. Maybe we should test this on a alza dataset which is way 
more structured.